## Write Weights, Biases and Scales to Text

!!! Note : This only works for the LeNet-5 Architecture and always need to be reworked if model changes because scales are stored in a semi ramdom way (?)

In [4]:
import onnx
from onnx import numpy_helper
import numpy as np

model = onnx.load("Lenet_qonnx.onnx")

layers = model.graph.initializer

layers_np = [numpy_helper.to_array(l) for l in layers]

# retrieve weight vectors
weights = [l for l in layers_np if len(np.shape(l)) >= 2]

# retrieve scale values
scales = [l for l in layers_np if len(np.shape(l)) == 0 and l > 0 and l < 1]

for w in weights:
    print(np.shape(w))

print("")

for s in scales:
    print(s)

(6, 1, 5, 5)
(16, 6, 5, 5)
(120, 400)
(84, 120)
(10, 84)

0.015625
0.0078125
0.0625
0.00390625
0.001953125
0.03125
0.125


### Creation and Exporting of quantized weight values

In [5]:
x = 0

weight_scales = [scales[1], scales[3], scales[4], scales[3], scales[1]]

for weight in weights:    
    weights_quant = [w * (1 / weight_scales[x]) for w in weight]
    weights_quant = [w.astype(np.uint8) for w in weights_quant]

    with open("../../data/layer_weight_" + str(x) + ".data", "w") as f:
        for i in np.reshape(weights_quant, -1):
            f.write(f'{i:02X}\n')
    
    x += 1

### Creation and Exporting of quantized scale values

In [6]:
scale_list = [(scales[0] * scales[1] / scales[2]), (scales[2] * scales[3] / scales[2]), (scales[2] * scales[4] / scales[0]), (scales[0] * scales[3] / scales[5]), (scales[5] * scales[1] / scales[6])]

scale_list = [np.log(s) / np.log(2) for s in scale_list]
scale_list = [np.trunc(s).astype(np.int8) for s in scale_list]
scale_list = [s * (-1) for s in scale_list]
scale_list = [s.astype(np.uint8) for s in scale_list]

with open("../../data/scale_values.data", "w") as f:
    for i in scale_list:
        f.write(f'{i:02X}\n')